# 4. アーキテクチャ

1-stage ERMを固定し、ResNetを基準に3つの条件付きモデルを比較する。

metadata利用とモデル容量が異なるため、metadataだけの効果とは呼ばない。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import rootutils
from IPython.display import display

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
from analysis.common.paths import STYLE_SHEET, study_dir  # noqa: E402
from analysis.archive.race_known_architecture_objective_comparison.comparison import (  # noqa: E402
    METRICS,
    evaluate,
    group_ids,
    paired_deltas,
    plot_performance,
    prediction_command,
    read_predictions,
    select_runs,
)

plt.style.use(STYLE_SHEET)
PACKAGE = study_dir("race_known_architecture_objective_comparison")
TOPIC = "04_architecture_comparison"
SEEDS = [42, 43, 44]
CONDITIONS = {
    "M01": "resnet-chexpert-race-known",
    "M03": "spatial-lora-chexpert-race-known",
    "M16": "spatial-lora-chexpert-stage1-4-race-known",
    "M07": "hyperadapt-chexpert-race-known",
}
LABELS = {"M01": "ResNet", "M03": "Spatial S34", "M16": "Spatial S14", "M07": "HyperAdapt"}
PAIRS = [["M01", "M03"], ["M01", "M16"], ["M01", "M07"]]
RESULTS = PACKAGE / "results" / TOPIC
FIGURES = PACKAGE / "figures" / TOPIC
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

## 入力

成功run・best checkpointと学習時manifestのtrain/val CSVをSHA256で照合する。testは共通CSVの現在のSHA256を記録し、cacheの行順・targetを検証する。cache欠損時は提示コマンドをrepo rootで実行（GPU自動起動なし）。

In [ ]:
provenance, frame, directories = select_runs(PACKAGE, CONDITIONS, SEEDS)
provenance.to_csv(RESULTS / "provenance.csv", index=False)
display(
    provenance[
        [
            "condition",
            "seed",
            "run_id",
            "checkpoint",
            "objective",
            "class_weight_scope",
            "freeze_backbone",
            "parent_run",
        ]
    ]
)
command = prediction_command(PACKAGE, directories, device="cuda:0")
print(command or "全予測cacheが有効")

## 全体性能・公平性

AUROC・worst-group AUROCが主指標。BACCとgapは補助。固定argmaxでEopp0=TNR gap、Eopp1=TPR gap、Eodds=(TPR gap+FPR gap)/2。平均±標準偏差は3seed間のばらつきで、有意差を意味しない。

In [ ]:
caches = read_predictions(PACKAGE, directories, frame)
summary, subgroup = evaluate(caches, frame)
summary.to_csv(RESULTS / "metrics_by_seed.csv", index=False)
subgroup.to_csv(RESULTS / "subgroup_by_seed.csv", index=False)
condition_stats = summary.groupby("condition", sort=False)[METRICS].agg(["mean", "std"])
condition_stats.to_csv(RESULTS / "condition_mean_std.csv")
display(condition_stats.round(4))

## 同seedの差

比較条件−baselineをseed別に示す。AUROC/BACCは正、gap/Eopp/Eoddsは負が改善方向。gap縮小はworst改善かbest低下かを併せて読む。

In [ ]:
delta = paired_deltas(summary, PAIRS)
delta.to_csv(RESULTS / "paired_deltas.csv", index=False)
delta_stats = delta.groupby("pair", sort=False)[METRICS].agg(["mean", "std"])
delta_stats.to_csv(RESULTS / "paired_delta_mean_std.csv")
display(delta.round(4))
display(delta_stats.round(4))

## 8群の性能

年齢<65／≥65 × White・Asian・Black・Others。件数・陽性/陰性数を添える。欠損属性は群評価のみ除外し、single-class群の指標はNaNのまま扱う。

In [ ]:
counts = subgroup[subgroup.condition == next(iter(CONDITIONS))].query("seed == 42")[
    ["group_id", "group", "n", "n_positive", "n_negative"]
]
counts.to_csv(RESULTS / "group_counts.csv", index=False)
print(f"test全体: {len(frame)}, 8群対象: {(group_ids(frame) >= 0).sum()}, 属性欠損除外: {(group_ids(frame) < 0).sum()}")
display(counts.set_index("group"))
group_stats = subgroup.groupby(["group", "condition"], sort=False)[["auroc", "bacc"]].agg(["mean", "std"])
group_stats.to_csv(RESULTS / "subgroup_mean_std.csv")
display(group_stats.round(4))
fig = plot_performance(summary, subgroup, LABELS)
fig.savefig(FIGURES / "performance.png", dpi=160)
plt.show()

## TODO

- DRO/G・DRO/SGの1-stageと凍結2-stageを別パネルで比較（analysis_plan.md）。容量・計算量を確認。
- 単独属性・患者単位の不確実性とq／群lossの詳細な対応を確認。3seedで方式の優劣を断定しない。